# 06 — Interface to the LLM backend measurements

**Question.** How will vLLM and llama.cpp operating points be placed on these measured rooflines, and where should they fall?

This notebook is design and preparation only: no backend was run for this analysis. It fixes the schema, the rule that picks the right ceiling for a point, and analytic guide lines showing where decode and prefill should land. Points drawn below are synthetic and labelled so; they exist to test the adapter.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/hwbench').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from hwbench import report, roofline, plotting
from hwbench.notebook import setup
ctx = setup('06_llm_roofline_interface')
thesis, ext, theory = ctx.thesis, ctx.ext, ctx.theory
table, figure, say = ctx.table, ctx.figure, ctx.say

## 1. Ceiling selection rule

A serving point is judged against the ceiling of the execution path its kernels actually use, at the core count and memory configuration of the server: 55 compute cores on socket 0 (one core is left for the frontend), Cache mode, memory bound to socket 0. That is why the ISA sweep includes 55 cores.

| Configuration | Weight GEMM path | Notes |
| --- | --- | --- |
| vLLM BF16 | AMX BF16 (oneDNN `brg_matmul:avx10_1_512_amx`) | verified here with oneDNN verbose |
| vLLM INT8 (W8A8) | AMX INT8 (`onednn.qlinear_pointwise`, u8×s8) | the only INT8 oneDNN path in the image that runs on AMX |
| llama.cpp BF16 | AVX-512 BF16 (`vdpbf16ps`) | no AMX BF16 kernel; `AMX_BUSY` = 0 |
| llama.cpp Q8_0 | AMX INT8 for ≥2 rows, VNNI at 1 row | |

In [ ]:
from hwbench.llm_adapter import CONFIGURATION_PATHS, MODEL_SPECS
rows = []
for cfg, (prec, path) in CONFIGURATION_PATHS.items():
    c = report.compute_ceiling(ext, path, 'cache', 55)
    rows.append(dict(configuration=cfg, precision=prec, path=path, cores=c.cores, TOPs_measured=c.gops / 1000, ghz_effective=c.ghz,
                     TOPs_theory_base=theory.peak_gops(path, 55) / 1000))
table(pd.DataFrame(rows))

## 2. Analytic guide lines

For Llama-3.1-8B (32 layers, hidden 4096, FFN 14336, 8 KV heads of dimension 128), one decode step at batch `B` performs about `2 × params × B` operations and reads the weights once, plus each sequence's KV cache. Prefill of `T` tokens performs `2 × params × T` operations against one read of the weights.

In [ ]:
spec = MODEL_SPECS['llama31_8b']
params = spec.linear_params + spec.hidden * spec.vocab
bytes_per_param = {'BF16 weights': 2.0, 'Q8_0 (34 B per 32 weights)': 34 / 32, 'W8A8 INT8': 1.0}
kv_per_token = 2 * spec.layers * spec.kv_heads * spec.head_dim * 2     # bytes, BF16 K and V
batch = np.array([1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 4096])
context = 1024
rows = []
for name, b in bytes_per_param.items():
    ops = 2 * params * batch
    traffic = params * b + batch * context * kv_per_token
    rows.append(pd.DataFrame(dict(weights=name, batch=batch, intensity=ops / traffic)))
guide = pd.concat(rows)
table(guide.pivot(index='batch', columns='weights', values='intensity').reset_index())

In [ ]:
mem = [report.memory_ceiling(ext, thesis, 'cache', 'effective_cache', resident_gib=16)]
cs = [report.compute_ceiling(ext, 'amx_bf16', 'cache', 55), report.compute_ceiling(ext, 'avx512_bf16', 'cache', 55),
      report.compute_ceiling(ext, 'amx_int8', 'cache', 55), report.compute_ceiling(ext, 'vnni_int8', 'cache', 55)]
fig = plotting.roofline_figure(cs, mem, title='Where LLM decode and prefill should fall · Cache · 55 cores · 16 GiB resident', x_range=(0.5, 8192))
for name, color in zip(bytes_per_param, ['#C24A40', '#268270', '#7A3E3E']):
    part = guide[guide.weights == name]
    fig.add_scatter(x=part.intensity, y=[min(cs[0].gops, mem[0].gbs * i) / 1000 for i in part.intensity], mode='markers+lines', name=f'decode guide · {name}',
                    line=dict(dash='dot', color=color), marker=dict(size=5, color=color))
fig.add_vrect(x0=128, x1=4096, fillcolor='#E8EDF2', opacity=0.5, line_width=0, annotation_text='prefill 128–4096 tokens', annotation_position='top left')
figure(fig, 'llm_guide_lines')

In [ ]:
ridge_bf16 = roofline.ridge(cs[0].gops, mem[0].gbs)
say(f"**Reading the guide.** Decode intensity equals the batch size for BF16 weights (one multiply-add per two weight bytes per sequence): at batch 1 it is about 1 operation per byte, "
    f"far left of the AMX BF16 ridge at {ridge_bf16:.0f} operations per byte, so decode is bandwidth-bound until batches reach the hundreds. Prefill of 128–4096 tokens crosses the ridge. "
    f"Q8_0 and W8A8 halve the weight traffic and double the intensity at the same batch.")

## 3. Adapter check on a synthetic fixture

`hwbench.llm_adapter.from_online_points` converts rows of `llm-backend-analysis/data/processed/online_points.csv` into workload points. The fixture below is **synthetic** (round numbers chosen to exercise the code); it is not a measurement.

In [ ]:
from hwbench.llm_adapter import from_online_points, to_frame
fixture = pd.DataFrame([
    dict(model='llama31_8b', configuration_id='vllm_bf16', backend='vllm', concurrency=1, isl=512, osl=512, output_tok_s_mean=15.0, mem_total_gbs_mean=300.0, amx_busy_frac_mean=0.05, ghz_effective_per_core_mean=2.2),
    dict(model='llama31_8b', configuration_id='llama_cpp_bf16', backend='llama_cpp', concurrency=1, isl=512, osl=512, output_tok_s_mean=12.0, mem_total_gbs_mean=250.0, amx_busy_frac_mean=0.0, ghz_effective_per_core_mean=2.2),
])
points = from_online_points(fixture)
table(to_frame(points)[['name', 'precision', 'isa_path', 'gops', 'intensity', 'bytes_source', 'cores', 'phase']])
fig = plotting.roofline_figure(cs, mem, points, title='SYNTHETIC fixture points on the Cache roofline (adapter test only)', x_range=(0.5, 8192))
figure(fig, 'adapter_fixture')

## 4. Methodology for real points

* **Prefill and decode are separate points.** A mixed server step blends two intensities; place prefill-only and decode-only windows separately, or label a point `mixed`.
* **Modeled operations, measured bytes.** Operations are `2 × (linear parameters + output head)` per token; attention over the KV cache is excluded and noted. Bytes come from the PMU's beyond-L3 counters (`pmu_l3miss`), which in Cache mode count traffic to the memory side as a whole; they are never mixed with modeled bytes in one series.
* **Placement matching.** The server's cores and memory binding decide the ceiling: 55 cores → the 55-core entries above; Cache mode → the resident or spilled slope according to the model's footprint (16 GiB of BF16 weights fit; a 70 GB model does not).
* **Flat-mode extension.** For a Flat comparison the weights would be bound with `numactl --membind` to HBM or DDR and judged against the Flat HBM or DDR slope of notebook 05.